## 1. Configuration et Imports

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.avro.functions import from_avro, to_avro
import json
import time
import os

In [ ]:
# Installation de kafka-python si nécessaire
# !pip install kafka-python

In [ ]:
# Création de la session Spark avec support Kafka et Delta Lake
spark = SparkSession.builder \
    .appName("IoT_Kafka_to_Delta_Silver") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 2. Configuration Kafka

Configuration du broker Kafka et du topic pour les données IoT

In [ ]:
# Configuration Kafka
KAFKA_BOOTSTRAP_SERVERS = "localhost:9092"  # Adapter selon votre configuration
KAFKA_TOPIC = "iot_sensors"

# Configuration des chemins
BASE_PATH = "/tmp/iot_kafka_pipeline"
SILVER_PATH = f"{BASE_PATH}/delta/silver/sensor_data"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints/silver_sensor_data"

# Création des répertoires
os.makedirs(os.path.dirname(SILVER_PATH), exist_ok=True)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)

print(f"Kafka Bootstrap Servers: {KAFKA_BOOTSTRAP_SERVERS}")
print(f"Kafka Topic: {KAFKA_TOPIC}")
print(f"Silver Path: {SILVER_PATH}")
print(f"Checkpoint Path: {CHECKPOINT_PATH}")

## 3. Producteur Kafka - Simulateur de Capteurs IoT

Création d'un producteur Kafka pour simuler l'envoi de données de capteurs

In [ ]:
from kafka import KafkaProducer
from kafka.errors import KafkaError
import json
import glob
import time
import threading

class IoTSensorSimulator:
    """Simulateur de capteurs IoT envoyant des données vers Kafka depuis les fichiers réels"""
    
    def __init__(self, bootstrap_servers, topic, data_folder="data/sensor_data"):
        self.bootstrap_servers = bootstrap_servers
        self.topic = topic
        self.data_folder = data_folder
        self.producer = None
        self.running = False
        
    def connect(self):
        """Connexion au cluster Kafka"""
        try:
            self.producer = KafkaProducer(
                bootstrap_servers=self.bootstrap_servers,
                value_serializer=lambda v: json.dumps(v).encode('utf-8'),
                acks='all',
                retries=3
            )
            print(f"✓ Connecté au broker Kafka: {self.bootstrap_servers}")
            return True
        except Exception as e:
            print(f"✗ Erreur de connexion à Kafka: {e}")
            return False
    
    def send_message(self, data):
        """Envoie un message vers Kafka"""
        try:
            future = self.producer.send(self.topic, value=data)
            record_metadata = future.get(timeout=10)
            return True
        except KafkaError as e:
            print(f"Erreur d'envoi: {e}")
            return False
    
    def stream_from_files(self, interval=0.1, max_messages=None):
        """Lit et envoie les données depuis les fichiers JSON réels"""
        self.running = True
        messages_sent = 0
        
        # Lister tous les fichiers JSON dans le dossier
        json_files = sorted(glob.glob(f"{self.data_folder}/*.json"))
        print(f"\n🚀 Démarrage du streaming depuis {len(json_files)} fichiers")
        print(f"   Intervalle: {interval}s entre messages\n")
        
        try:
            for json_file in json_files:
                if not self.running:
                    break
                    
                with open(json_file, 'r') as f:
                    for line in f:
                        if not self.running:
                            break
                        
                        try:
                            data = json.loads(line.strip())
                            if self.send_message(data):
                                messages_sent += 1
                                if messages_sent % 50 == 0:
                                    print(f"📤 {messages_sent} messages envoyés (device: {data.get('device_id', 'N/A')}, building: {data.get('building', 'N/A')})")
                                
                                if max_messages and messages_sent >= max_messages:
                                    print(f"\n✓ Limite de {max_messages} messages atteinte")
                                    self.running = False
                                    break
                                    
                                time.sleep(interval)
                        except json.JSONDecodeError as e:
                            print(f"Erreur de parsing JSON: {e}")
                            continue
                            
        except KeyboardInterrupt:
            print("\n⏸️  Arrêt demandé par l'utilisateur")
        finally:
            self.running = False
            print(f"\n✓ Streaming terminé. Total envoyé: {messages_sent} messages")
    
    def stop(self):
        """Arrête le simulateur"""
        self.running = False
        if self.producer:
            self.producer.flush()
            self.producer.close()
            print("Producer Kafka fermé")

# Création du simulateur avec les données réelles
simulator = IoTSensorSimulator(KAFKA_BOOTSTRAP_SERVERS, KAFKA_TOPIC)
print("Simulateur créé (utilise les données de data/sensor_data/)")

## 4. Démarrage du Producteur (en arrière-plan)

⚠️ **Note** : Assurez-vous que Kafka est démarré avant d'exécuter cette cellule

Pour démarrer Kafka localement (exemples) :
```bash
# Zookeeper
bin/zookeeper-server-start.sh config/zookeeper.properties

# Kafka
bin/kafka-server-start.sh config/server.properties

# Créer le topic
bin/kafka-topics.sh --create --topic iot_sensors --bootstrap-server localhost:9092 --partitions 3 --replication-factor 1
```

In [ ]:
# Connexion au broker Kafka
if simulator.connect():
    # Démarrer le producteur dans un thread séparé
    producer_thread = threading.Thread(
        target=simulator.stream_from_files,
        kwargs={"interval": 0.1, "max_messages": 500}  # 0.1s entre messages, max 500 messages
    )
    producer_thread.daemon = True
    producer_thread.start()
    print("\n✓ Producteur démarré en arrière-plan (streaming depuis data/sensor_data/)")
else:
    print("\n✗ Impossible de démarrer le producteur - vérifiez que Kafka est en cours d'exécution")

## 5. Schéma des Données Silver

Schéma enrichi pour la couche Silver avec champs supplémentaires

In [ ]:
# Schéma des données JSON dans Kafka (basé sur les données réelles)
sensor_schema = StructType([
    StructField("timestamp", StringType(), False),
    StructField("device_id", StringType(), False),
    StructField("building", StringType(), False),
    StructField("floor", IntegerType(), False),
    StructField("type", StringType(), False),
    StructField("value", DoubleType(), False),
    StructField("unit", StringType(), False)
])

print("Schéma défini (basé sur les données réelles de data/sensor_data/)")

## 6. Lecture du Flux Kafka

Configuration du consommateur Spark Structured Streaming pour Kafka

In [ ]:
# Lecture du flux Kafka
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .option("maxOffsetsPerTrigger", 100) \
    .option("failOnDataLoss", "false") \
    .load()

print("Flux Kafka configuré")
print("Schéma Kafka:")
kafka_stream.printSchema()

## 7. Parsing et Transformations Silver

Transformations avancées pour la couche Silver :
- Parsing du JSON depuis Kafka
- Conversion des types
- Enrichissement avec métadonnées Kafka (partition, offset, timestamp)
- Calculs dérivés (confort thermique, qualité de l'air)
- Normalisation des données

In [ ]:
# Parsing du JSON et application des transformations Silver
silver_stream = kafka_stream \
    .select(
        col("topic"),
        col("partition"),
        col("offset"),
        col("timestamp").alias("kafka_timestamp"),
        from_json(col("value").cast("string"), sensor_schema).alias("data")
    ) \
    .select(
        col("data.*"),
        col("partition"),
        col("offset"),
        col("kafka_timestamp")
    ) \
    .filter(col("device_id").isNotNull()) \
    .filter(col("building").isNotNull()) \
    .withColumn("event_timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("processing_time", current_timestamp()) \
    .withColumn(
        "comfort_index",
        when(
            (col("type") == "temperature") & (col("value").between(20, 24)),
            "comfortable"
        ).when(
            (col("type") == "temperature") & (col("value").between(18, 26)),
            "acceptable"
        ).otherwise("N/A")
    ) \
    .withColumn(
        "air_quality",
        when((col("type") == "co2") & (col("value") <= 600), "excellent") \
        .when((col("type") == "co2") & (col("value") <= 800), "good") \
        .when((col("type") == "co2") & (col("value") <= 1000), "fair") \
        .when((col("type") == "co2"), "poor") \
        .otherwise("N/A")
    ) \
    .withColumn(
        "anomaly_detected",
        when((col("type") == "co2") & (col("value") > 1000), True)
        .when((col("type") == "temperature") & ((col("value") < 15) | (col("value") > 30)), True)
        .when((col("type") == "humidity") & ((col("value") < 20) | (col("value") > 80)), True)
        .otherwise(False)
    ) \
    .withColumn(
        "data_quality_flag",
        when(
            col("device_id").isNotNull() & 
            col("value").isNotNull() & 
            col("building").isNotNull(),
            "complete"
        ).otherwise("partial")
    ) \
    .select(
        col("device_id"),
        col("building"),
        col("floor"),
        col("type"),
        col("value"),
        col("unit"),
        col("event_timestamp"),
        col("anomaly_detected"),
        col("comfort_index"),
        col("air_quality"),
        col("data_quality_flag"),
        col("partition").alias("kafka_partition"),
        col("offset").alias("kafka_offset"),
        col("kafka_timestamp"),
        col("processing_time")
    )

print("Transformations Silver configurées")
print("\nSchéma Silver:")
silver_stream.printSchema()

## 8. Écriture dans Delta Lake (Silver)

Configuration de l'écriture en streaming vers Delta Lake Silver :
- Mode `append` pour ajouter les nouvelles données
- Checkpoint pour la tolérance aux pannes
- Trigger toutes les 5 secondes
- Garantie exactly-once grâce aux offsets Kafka + checkpoint Spark

In [ ]:
# Écriture en streaming vers Delta Lake Silver
query = silver_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime="5 seconds") \
    .start(SILVER_PATH)

print(f"\n✓ Streaming query démarré: {query.name}")
print(f"Query ID: {query.id}")
print(f"Status: {query.status}")
print(f"\n🔄 Le pipeline consomme maintenant les messages Kafka...")

## 9. Surveillance du Pipeline

Monitoring des métriques du streaming

In [ ]:
# Attendre quelques secondes pour assurer le traitement
time.sleep(10)

# Lire les données Silver en mode batch
silver_df = spark.read.format("delta").load(SILVER_PATH)

print(f"\n=== DONNÉES SILVER ===")
print(f"Nombre total d'enregistrements: {silver_df.count()}")
print("\nAperçu des données:")
silver_df.select(
    "device_id", "building", "floor", "type", "value", 
    "event_timestamp", "anomaly_detected", "comfort_index", "air_quality"
).show(10, truncate=False)

## 10. Vérification des Données Silver

Lecture batch des données pour analyse

In [ ]:
# Attendre quelques secondes pour assurer le traitement
time.sleep(10)

# Lire les données Silver en mode batch
silver_df = spark.read.format("delta").load(SILVER_PATH)

print(f"\n=== DONNÉES SILVER ===")
print(f"Nombre total d'enregistrements: {silver_df.count()}")
print("\nAperçu des données:")
silver_df.select(
    "sensor_id", "building_id", "event_timestamp", 
    "temperature", "humidity", "comfort_index", 
    "air_quality", "anomaly_detected"
).show(10, truncate=False)

In [ ]:
# Statistiques par bâtiment
print("\n=== STATISTIQUES PAR BÂTIMENT ===")
silver_df.groupBy("building").agg(
    count("*").alias("records"),
    countDistinct("device_id").alias("unique_sensors"),
    sum(when(col("anomaly_detected"), 1).otherwise(0)).alias("anomalies")
).orderBy("building").show()

In [ ]:
# Distribution par type de capteur
print("\n=== STATISTIQUES PAR TYPE DE CAPTEUR ===")
silver_df.groupBy("type").agg(
    count("*").alias("records"),
    avg("value").alias("avg_value"),
    min("value").alias("min_value"),
    max("value").alias("max_value")
).show()

In [ ]:
# Analyse des anomalies
print("\n=== ANALYSE DES ANOMALIES ===")
anomalies_df = silver_df.filter(col("anomaly_detected") == True)
print(f"Nombre total d'anomalies détectées: {anomalies_df.count()}")

if anomalies_df.count() > 0:
    print("\nDernières anomalies:")
    anomalies_df.select(
        "sensor_id", "building_id", "event_timestamp",
        "temperature", "humidity", "co2_level", "comfort_index", "air_quality"
    ).orderBy(desc("event_timestamp")).show(5, truncate=False)

## 11. Analyse des Offsets et Partitions Kafka

Compréhension des offsets et partitions consommés

In [ ]:
# Analyse des métadonnées Kafka
print("\n=== MÉTADONNÉES KAFKA ===")
print("\nDistribution par partition:")
silver_df.groupBy("kafka_partition").agg(
    count("*").alias("messages_count"),
    min("kafka_offset").alias("min_offset"),
    max("kafka_offset").alias("max_offset")
).orderBy("kafka_partition").show()

print("\nPlage d'offsets par partition (montre le parallélisme):")
silver_df.groupBy("kafka_partition").agg(
    (max("kafka_offset") - min("kafka_offset") + 1).alias("offset_range")
).show()

## 12. Latence du Pipeline

Mesure de la latence entre l'événement et le traitement

In [ ]:
# Calcul de la latence de traitement
print("\n=== LATENCE DU PIPELINE ===")

latency_df = silver_df.withColumn(
    "latency_seconds",
    (unix_timestamp(col("processing_time")) - unix_timestamp(col("event_timestamp")))
)

latency_stats = latency_df.agg(
    avg("latency_seconds").alias("avg_latency"),
    min("latency_seconds").alias("min_latency"),
    max("latency_seconds").alias("max_latency"),
    percentile_approx("latency_seconds", 0.5).alias("median_latency"),
    percentile_approx("latency_seconds", 0.95).alias("p95_latency")
).collect()[0]

print(f"Latence moyenne: {latency_stats['avg_latency']:.2f} secondes")
print(f"Latence minimale: {latency_stats['min_latency']:.2f} secondes")
print(f"Latence maximale: {latency_stats['max_latency']:.2f} secondes")
print(f"Latence médiane: {latency_stats['median_latency']:.2f} secondes")
print(f"Latence P95: {latency_stats['p95_latency']:.2f} secondes")

## 13. Historique Delta Lake

Vérification de l'historique des transactions

In [ ]:
from delta.tables import DeltaTable

# Historique des transactions Delta
delta_table = DeltaTable.forPath(spark, SILVER_PATH)
print("\n=== HISTORIQUE DELTA LAKE ===")
delta_table.history().select(
    "version", "timestamp", "operation", "operationMetrics"
).show(10, truncate=False)

## 14. Arrêt du Pipeline

In [ ]:
# Arrêter le streaming query
print("\n⏹️  Arrêt du streaming query...")
query.stop()
time.sleep(2)
print(f"Is Active: {query.isActive}")
print("✓ Streaming query arrêté")

In [ ]:
# Arrêter le producteur Kafka
print("\n⏹️  Arrêt du producteur Kafka...")
simulator.stop()
print("✓ Producteur arrêté")

## 15. Analyse Finale et Insights

In [ ]:
# Statistiques finales
final_df = spark.read.format("delta").load(SILVER_PATH)

print("\n" + "="*60)
print("=== RAPPORT FINAL ===")
print("="*60)

total_records = final_df.count()
print(f"\n📊 Total d'enregistrements traités: {total_records}")

# Période couverte
time_range = final_df.agg(
    min("event_timestamp").alias("first_event"),
    max("event_timestamp").alias("last_event")
).collect()[0]

print(f"⏱️  Premier événement: {time_range['first_event']}")
print(f"⏱️  Dernier événement: {time_range['last_event']}")

# Capteurs et bâtiments uniques
unique_sensors = final_df.select("sensor_id").distinct().count()
unique_buildings = final_df.select("building_id").distinct().count()

print(f"\n🏢 Bâtiments uniques: {unique_buildings}")
print(f"📡 Capteurs uniques: {unique_sensors}")

# Qualité des données
data_quality = final_df.groupBy("data_quality_flag").count().collect()
print("\n✅ Qualité des données:")
for row in data_quality:
    percentage = (row['count'] / total_records) * 100
    print(f"   {row['data_quality_flag']}: {row['count']} ({percentage:.1f}%)")

# Taux d'anomalies
anomaly_count = final_df.filter(col("anomaly_detected") == True).count()
anomaly_rate = (anomaly_count / total_records) * 100 if total_records > 0 else 0
print(f"\n⚠️  Anomalies détectées: {anomaly_count} ({anomaly_rate:.2f}%)")

print("\n" + "="*60)

## 16. Nettoyage (Optionnel)

In [ ]:
# Décommenter pour nettoyer les données de test
# import shutil
# shutil.rmtree(BASE_PATH, ignore_errors=True)
# print(f"Répertoire {BASE_PATH} supprimé")

---

## Concepts Clés Illustrés

### 1. **Apache Kafka - Message Broker**

#### Rôle dans l'Architecture Temps Réel
- **Découplage** : Les producteurs (capteurs) et consommateurs (Spark) sont indépendants
- **Scalabilité horizontale** : Ajout de brokers et partitions pour gérer la charge
- **Persistance** : Messages conservés même après consommation (durée configurable)
- **Haute disponibilité** : Réplication des données sur plusieurs brokers
- **Buffer temporaire** : Absorption des pics de charge sans perte de données

#### **Offsets**
- Position unique d'un message dans une partition
- Séquence monotone incrémentale (0, 1, 2, ...)
- Permet la reprise exacte en cas d'échec
- Garantie exactly-once avec Spark checkpointing

#### **Partitions**
- Division logique d'un topic pour parallélisme
- Chaque partition est une séquence ordonnée de messages
- Permet la scalabilité horizontale de la lecture/écriture
- Distribution basée sur une clé (ou round-robin)
- Ordre garanti au niveau partition (pas entre partitions)

#### **Consumer Groups**
- Groupe de consommateurs partageant la lecture d'un topic
- Chaque partition est lue par un seul consommateur du groupe
- Permet le parallélisme et la haute disponibilité
- Rééquilibrage automatique en cas de panne d'un consommateur

### 2. **Spark Structured Streaming avec Kafka**

#### Intégration Kafka
- Consommation native des topics Kafka
- Gestion automatique des offsets
- Support du parallélisme (1 task Spark par partition Kafka)

#### Garanties de Livraison
- **Exactly-once** : Combinaison offsets Kafka + checkpoint Spark + transactions Delta
- Idempotence des opérations
- Récupération automatique après échec

### 3. **Architecture Médaillon - Silver**

#### Caractéristiques Silver
- Données nettoyées et normalisées
- Enrichissement avec calculs métier
- Dédoublonnage et validation de qualité
- Schéma structuré et typé
- Prête pour l'analyse et les tableaux de bord

#### Transformations Appliquées
1. **Parsing** : Extraction du JSON depuis Kafka
2. **Typage** : Conversion vers types appropriés
3. **Validation** : Filtrage des valeurs invalides
4. **Enrichissement** : Calculs dérivés (comfort_index, air_quality)
5. **Normalisation** : Standardisation des formats
6. **Métadonnées** : Ajout timestamps et informations Kafka

### 4. **Monitoring et Observabilité**

#### Métriques Clés
- **Input/Process Rate** : Débit d'ingestion et traitement
- **Latency** : Délai entre événement et traitement
- **Offsets Progress** : Avancement dans les partitions Kafka
- **Data Quality** : Taux de complétude et validité

### 5. **Delta Lake Avantages**
- **ACID** : Transactions atomiques
- **Versioning** : Historique complet des modifications
- **Time Travel** : Requêtes sur versions antérieures
- **Schema Evolution** : Gestion des changements de schéma
- **Upserts/Deletes** : Support des opérations complexes

### 6. **Cas d'Usage IoT**
- Détection d'anomalies en temps réel
- Monitoring du confort thermique
- Optimisation énergétique
- Alertes sur qualité de l'air
- Analytics temps réel pour tableaux de bord